In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# Generate the source data (reuses your existing generator, no Snowflake dependency)
import sys
sys.path.append("/Workspace/Users/youngli.hong2@gmail.com/restaurant_analytics")
from generate_restaurant_data import main
main()

  restaurants.csv          4 rows
  menu_items.csv          38 rows
  customers.csv          800 rows
  orders.csv           5,020 rows
  order_items.csv     13,088 rows

Wrote 5 files to ./raw_data/
  date range : 2024-01-01 → 2024-12-31
  guests     : ~18% of orders have no customer_id
  price bump : 108% from 2024-07-01
  messiness  : ON


In [0]:
# COMMAND ----------
# Load order_items.csv into a Spark DataFrame
order_items = spark.read.csv(
    "file:/Workspace/Users/youngli.hong2@gmail.com/restaurant_analytics/raw_data/order_items.csv",
    header=True,
    inferSchema=True,
)
order_items.show(5)


+-------------+--------+------------+--------+----------+
|order_item_id|order_id|menu_item_id|quantity|unit_price|
+-------------+--------+------------+--------+----------+
|            1|       1|          22|       3|       5.0|
|            2|       1|          27|       1|      10.0|
|            3|       1|          23|       1|       6.0|
|            4|       2|           4|       1|      12.0|
|            5|       2|          31|       1|       3.0|
+-------------+--------+------------+--------+----------+
only showing top 5 rows


In [0]:
# COMMAND ----------
from pyspark.sql.functions import col

# This is the direct PySpark port of int_order_items_priced.sql:
# revenue = quantity * unit_price, using the SNAPSHOTTED unit_price already
# on each row — deliberately not joined back to menu_items.base_price,
# since that would give current price, not transaction-time price.
priced = order_items.withColumn(
    "revenue", col("quantity") * col("unit_price")
)

priced.select("order_item_id", "order_id", "menu_item_id", "quantity", "unit_price", "revenue").show(10)


+-------------+--------+------------+--------+----------+-------+
|order_item_id|order_id|menu_item_id|quantity|unit_price|revenue|
+-------------+--------+------------+--------+----------+-------+
|            1|       1|          22|       3|       5.0|   15.0|
|            2|       1|          27|       1|      10.0|   10.0|
|            3|       1|          23|       1|       6.0|    6.0|
|            4|       2|           4|       1|      12.0|   12.0|
|            5|       2|          31|       1|       3.0|    3.0|
|            6|       2|          38|       3|      13.0|   39.0|
|            7|       3|          18|       1|      16.0|   16.0|
|            8|       3|          17|       1|      19.0|   19.0|
|            9|       4|          10|       1|      24.0|   24.0|
|           10|       4|          29|       1|       7.0|    7.0|
+-------------+--------+------------+--------+----------+-------+
only showing top 10 rows


In [0]:
# COMMAND ----------
# Sanity check: total revenue should match what dbt/Snowflake computed
total_revenue = priced.agg({"revenue": "sum"}).collect()[0][0]
print(f"Total revenue (PySpark): {total_revenue:,.2f}")

Total revenue (PySpark): 180,604.30
